# Audio Signal Processing for Machine Learning

In this notebook, we'll explore fundamental audio signal processing concepts needed for machine learning on audio data.

**What we'll learn:**
- Why audio is fundamentally different from images and text
- How to represent audio in time and frequency domains
- Audio features: spectrograms, mel spectrograms, and MFCCs
- Data augmentation techniques for audio
- Building a simple audio classifier

**Applications**: Speech recognition, music generation, audio classification, speaker identification, emotion detection

## 1. Setup

Let's import our dependencies and configure the environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import IPython.display as ipd
from scipy import signal
from aiml_notebooks import set_seed, get_device

set_seed(42)
device = get_device()
print(f"Using device: {device}")

## 2. Understanding Audio Signals

**Audio vs. Images vs. Text:**

- **Images**: 2D spatial data (height × width × channels), features are spatially local
- **Text**: 1D discrete sequence, features are symbolic tokens
- **Audio**: 1D continuous signal, features span both time AND frequency

**Key insight**: Audio is naturally a time-domain signal, but humans perceive frequency content (pitch, timbre). We need representations that capture both!

### Loading Sample Audio

We'll load a sample audio file from librosa's built-in examples.

In [ ]:
# Load a sample audio file (trumpet)
audio_path = librosa.example('trumpet')
y, sr = librosa.load(audio_path, duration=3.0)  # Load 3 seconds

print(f"Audio shape: {y.shape}")
print(f"Sample rate: {sr} Hz")
print(f"Duration: {len(y) / sr:.2f} seconds")
print(f"Value range: [{y.min():.3f}, {y.max():.3f}]")

### Listen to the Audio

Let's hear what this sounds like!

In [ ]:
# Play the audio
ipd.Audio(y, rate=sr)

## 3. Digital Audio Basics

**Digital audio** is a sampled representation of continuous sound waves.

**Key concepts:**
- **Sample rate (sr)**: How many samples per second (Hz). CD quality = 44,100 Hz
- **Bit depth**: Precision of each sample (16-bit, 24-bit, etc.)
- **Waveform**: Time-domain representation of amplitude over time

**Nyquist theorem**: To capture a frequency f, you need sample rate ≥ 2f. 
- Human hearing: ~20 Hz to 20,000 Hz
- So 44,100 Hz captures up to 22,050 Hz (enough for humans)

### Visualizing the Waveform

Let's plot the raw audio signal in the time domain.

In [ ]:
plt.figure(figsize=(14, 4))
librosa.display.waveshow(y, sr=sr, alpha=0.8)
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.title('Audio Waveform (Time Domain)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The waveform shows amplitude variations over time.")
print("But we can't easily see pitch, harmony, or timbre from this view!")

### Creating a Simple Tone

Let's generate a pure sine wave to understand audio fundamentals.

In [ ]:
# Generate a 440 Hz sine wave (musical note A4)
duration = 1.0  # seconds
sample_rate = 22050
frequency = 440  # Hz

t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
sine_wave = 0.5 * np.sin(2 * np.pi * frequency * t)

print(f"Generated {frequency} Hz sine wave")
print(f"Samples: {len(sine_wave)}")
print(f"Duration: {duration} seconds")

### Visualizing the Sine Wave

Plot a small portion to see the periodic oscillation.

In [ ]:
# Plot first 50ms
samples_to_plot = int(0.05 * sample_rate)

plt.figure(figsize=(14, 4))
plt.plot(t[:samples_to_plot], sine_wave[:samples_to_plot], linewidth=2)
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.title(f'{frequency} Hz Sine Wave (First 50ms)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"One complete cycle = 1/{frequency} = {1/frequency:.4f} seconds")

### Listen to the Pure Tone

This is what a 440 Hz tone sounds like.

In [ ]:
ipd.Audio(sine_wave, rate=sample_rate)

## 4. Fourier Transform: From Time to Frequency

**The problem**: Waveforms show amplitude over time, but not frequency content.

**The solution**: **Fourier Transform** decomposes a signal into its frequency components.

**Key insight**: Any signal can be represented as a sum of sine waves at different frequencies!

**DFT Formula**: For signal $x[n]$ of length $N$:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-i 2\pi k n / N}$$

Where $X[k]$ is the frequency component at frequency $k \cdot f_s / N$.

### Implementing the Discrete Fourier Transform

Let's implement a simple DFT and apply it to our sine wave.

In [ ]:
# Compute FFT (Fast Fourier Transform - efficient DFT algorithm)
fft_result = np.fft.fft(sine_wave)

# Get magnitude spectrum
magnitude = np.abs(fft_result)

# Get frequency bins
freqs = np.fft.fftfreq(len(sine_wave), 1/sample_rate)

print(f"FFT output shape: {fft_result.shape}")
print(f"Frequency bins: {len(freqs)}")
print(f"Frequency range: {freqs.min():.1f} to {freqs.max():.1f} Hz")

### Visualizing the Frequency Spectrum

Plot the magnitude spectrum to see the frequency content.

In [ ]:
# Only plot positive frequencies (spectrum is symmetric)
positive_freqs = freqs[:len(freqs)//2]
positive_magnitude = magnitude[:len(magnitude)//2]

plt.figure(figsize=(14, 4))
plt.plot(positive_freqs, positive_magnitude, linewidth=2)
plt.xlabel('Frequency (Hz)', fontsize=12)
plt.ylabel('Magnitude', fontsize=12)
plt.title('Frequency Spectrum of 440 Hz Sine Wave', fontsize=14)
plt.xlim(0, 1000)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice the spike at 440 Hz!")
print("This confirms our sine wave is a pure tone at that frequency.")

### Fourier Transform on Real Audio

Now let's see what the trumpet's frequency spectrum looks like.

In [ ]:
# Compute FFT of trumpet audio
fft_trumpet = np.fft.fft(y)
magnitude_trumpet = np.abs(fft_trumpet)
freqs_trumpet = np.fft.fftfreq(len(y), 1/sr)

# Plot positive frequencies
positive_freqs = freqs_trumpet[:len(freqs_trumpet)//2]
positive_magnitude = magnitude_trumpet[:len(magnitude_trumpet)//2]

plt.figure(figsize=(14, 4))
plt.plot(positive_freqs, positive_magnitude, linewidth=1, alpha=0.8)
plt.xlabel('Frequency (Hz)', fontsize=12)
plt.ylabel('Magnitude', fontsize=12)
plt.title('Frequency Spectrum of Trumpet Audio', fontsize=14)
plt.xlim(0, 5000)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Real audio has many frequency components!")
print("But this is the AVERAGE over the entire clip - we lose time information.")

## 5. Short-Time Fourier Transform (STFT)

**Problem**: Standard FFT loses time information - we can't tell WHEN frequencies occur.

**Solution**: **Short-Time Fourier Transform (STFT)**
- Divide signal into short overlapping windows
- Apply FFT to each window
- Get time-frequency representation!

**Parameters**:
- **Window size (n_fft)**: Frequency resolution vs. time resolution tradeoff
  - Larger window → better frequency resolution, worse time resolution
  - Smaller window → better time resolution, worse frequency resolution
- **Hop length**: How much to shift between windows (overlap = n_fft - hop_length)

### Computing the STFT

Let's compute the STFT and create a spectrogram.

In [ ]:
# STFT parameters
n_fft = 2048  # Window size
hop_length = 512  # Samples between windows

# Compute STFT
D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)

# Get magnitude (discard phase)
magnitude = np.abs(D)

print(f"STFT shape: {D.shape}")
print(f"Frequency bins: {D.shape[0]}")
print(f"Time frames: {D.shape[1]}")
print(f"Time resolution: {hop_length / sr * 1000:.1f} ms per frame")
print(f"Frequency resolution: {sr / n_fft:.1f} Hz per bin")

### Visualizing the Spectrogram

A **spectrogram** shows frequency content over time - it's a 2D image!

In [ ]:
# Convert to dB scale for better visualization
db_spectrogram = librosa.amplitude_to_db(magnitude, ref=np.max)

plt.figure(figsize=(14, 6))
librosa.display.specshow(db_spectrogram, sr=sr, hop_length=hop_length,
                        x_axis='time', y_axis='hz', cmap='viridis')
plt.colorbar(format='%+2.0f dB', label='Amplitude (dB)')
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('Frequency (Hz)', fontsize=12)
plt.title('Spectrogram (Linear Frequency Scale)', fontsize=14)
plt.ylim(0, 8000)
plt.tight_layout()
plt.show()

print("Now we can see BOTH time AND frequency!")
print("Brighter = higher amplitude at that time-frequency location.")

### Understanding Window Size Tradeoffs

Let's compare different window sizes to see the time-frequency resolution tradeoff.

In [ ]:
# Different window sizes
window_sizes = [512, 2048, 8192]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, n_fft in zip(axes, window_sizes):
    D = librosa.stft(y, n_fft=n_fft, hop_length=n_fft//4)
    db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    
    librosa.display.specshow(db, sr=sr, hop_length=n_fft//4,
                            x_axis='time', y_axis='hz', cmap='viridis', ax=ax)
    ax.set_ylim(0, 8000)
    ax.set_title(f'n_fft={n_fft}\nFreq res: {sr/n_fft:.1f} Hz', fontsize=11)
    ax.set_xlabel('Time (s)', fontsize=10)
    ax.set_ylabel('Frequency (Hz)', fontsize=10)

plt.suptitle('Window Size Effects on Spectrograms', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Small window (512): Good time resolution, blurry frequencies")
print("Large window (8192): Sharp frequencies, blurry time")
print("Medium window (2048): Balanced tradeoff")

## 6. Mel Scale: Human Perception of Frequency

**Problem**: Humans don't perceive frequency linearly!
- We're more sensitive to differences at low frequencies
- 100 Hz vs 200 Hz sounds like a bigger change than 10,000 Hz vs 10,100 Hz

**Solution**: **Mel scale** approximates human perception
- Low frequencies: fine resolution
- High frequencies: coarse resolution

**Formula** (one common approximation):

$$m = 2595 \cdot \log_{10}\left(1 + \frac{f}{700}\right)$$

Where $f$ is frequency in Hz and $m$ is frequency in mels.

### Visualizing the Mel Scale

Let's see how Hz maps to mels.

In [ ]:
# Convert Hz to mels
hz = np.linspace(0, 8000, 1000)
mels = librosa.hz_to_mel(hz)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(hz, mels, linewidth=2)
plt.xlabel('Frequency (Hz)', fontsize=12)
plt.ylabel('Frequency (Mels)', fontsize=12)
plt.title('Hz to Mel Conversion', fontsize=13)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(mels, hz, linewidth=2, color='orange')
plt.xlabel('Frequency (Mels)', fontsize=12)
plt.ylabel('Frequency (Hz)', fontsize=12)
plt.title('Mel to Hz Conversion', fontsize=13)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice the mel scale is approximately linear below ~1000 Hz")
print("and logarithmic above that - matching human perception!")

### Mel Filterbanks

We create **mel filterbanks** - triangular filters spaced on the mel scale.

In [ ]:
# Create mel filterbank
n_mels = 128  # Number of mel bands
mel_filters = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels)

print(f"Mel filterbank shape: {mel_filters.shape}")
print(f"({n_mels} mel bands × {n_fft//2 + 1} frequency bins)")

### Visualizing Mel Filterbanks

Each row is a triangular filter on the mel scale.

In [ ]:
plt.figure(figsize=(14, 6))

# Plot all filters as heatmap
plt.subplot(2, 1, 1)
librosa.display.specshow(mel_filters, sr=sr, x_axis='linear', cmap='viridis')
plt.ylabel('Mel Filter', fontsize=11)
plt.xlabel('Frequency (Hz)', fontsize=11)
plt.title('Mel Filterbank (All Filters)', fontsize=13)
plt.colorbar(format='%.2f')

# Plot a few individual filters
plt.subplot(2, 1, 2)
freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
for i in [0, 32, 64, 96, 127]:
    plt.plot(freqs, mel_filters[i], label=f'Mel band {i}', alpha=0.7)
plt.xlabel('Frequency (Hz)', fontsize=11)
plt.ylabel('Filter Amplitude', fontsize=11)
plt.title('Individual Mel Filters', fontsize=13)
plt.legend(fontsize=9)
plt.xlim(0, 8000)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice: Filters are narrower at low frequencies, wider at high frequencies.")
print("This matches human perception!")

### Creating a Mel Spectrogram

Apply mel filterbanks to the spectrogram to get mel-frequency representation.

In [ ]:
# Compute mel spectrogram
mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, 
                                         hop_length=hop_length, n_mels=n_mels)

# Convert to dB
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

print(f"Mel spectrogram shape: {mel_spec.shape}")
print(f"({n_mels} mel bands × {mel_spec.shape[1]} time frames)")

### Visualizing Mel Spectrogram

Compare linear frequency spectrogram vs. mel spectrogram.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Linear frequency spectrogram
D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
db_spec = librosa.amplitude_to_db(np.abs(D), ref=np.max)
librosa.display.specshow(db_spec, sr=sr, hop_length=hop_length,
                        x_axis='time', y_axis='hz', cmap='viridis', ax=axes[0])
axes[0].set_ylim(0, 8000)
axes[0].set_title('Linear Frequency Spectrogram', fontsize=13)
axes[0].set_xlabel('Time (s)', fontsize=11)
axes[0].set_ylabel('Frequency (Hz)', fontsize=11)

# Mel spectrogram
librosa.display.specshow(mel_spec_db, sr=sr, hop_length=hop_length,
                        x_axis='time', y_axis='mel', cmap='viridis', ax=axes[1])
axes[1].set_title('Mel Spectrogram', fontsize=13)
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_ylabel('Frequency (Mel)', fontsize=11)

plt.tight_layout()
plt.show()

print("Mel spectrogram has more detail at low frequencies (where we're sensitive)")
print("and less at high frequencies (where we're less sensitive).")

## 7. MFCCs: Mel-Frequency Cepstral Coefficients

**MFCCs** are one of the most popular audio features for machine learning!

**The idea**: 
1. Compute mel spectrogram (time-frequency in mel scale)
2. Take logarithm (humans perceive loudness logarithmically)
3. Apply **Discrete Cosine Transform (DCT)** to decorrelate and compress
4. Keep the first 13-40 coefficients

**Why DCT?** 
- Mel bands are correlated (neighboring frequencies often move together)
- DCT produces uncorrelated features
- Low DCT coefficients = overall spectral shape (timbre)
- High DCT coefficients = fine detail (usually discarded)

**Use cases**: Speech recognition, speaker identification, music classification

### Computing MFCCs

Extract MFCC features from the audio.

In [ ]:
# Compute MFCCs
n_mfcc = 13  # Typical: 13 or 20 coefficients
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)

print(f"MFCC shape: {mfccs.shape}")
print(f"({n_mfcc} coefficients × {mfccs.shape[1]} time frames)")
print(f"\nMFCC value range: [{mfccs.min():.1f}, {mfccs.max():.1f}]")

### Visualizing MFCCs

Plot the MFCC features over time.

In [ ]:
plt.figure(figsize=(14, 6))
librosa.display.specshow(mfccs, sr=sr, hop_length=hop_length, 
                        x_axis='time', cmap='coolwarm')
plt.colorbar(label='MFCC Value')
plt.ylabel('MFCC Coefficient', fontsize=12)
plt.xlabel('Time (s)', fontsize=12)
plt.title('MFCCs (Mel-Frequency Cepstral Coefficients)', fontsize=14)
plt.tight_layout()
plt.show()

print("First coefficient (MFCC-0): Overall energy")
print("Coefficients 1-12: Spectral shape features")
print("Higher coefficients capture finer spectral details")

### MFCC Statistics

Often we compute statistics (mean, std) over time for classification.

In [ ]:
# Compute mean and std of each MFCC over time
mfcc_mean = np.mean(mfccs, axis=1)
mfcc_std = np.std(mfccs, axis=1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.bar(range(n_mfcc), mfcc_mean)
plt.xlabel('MFCC Coefficient', fontsize=11)
plt.ylabel('Mean Value', fontsize=11)
plt.title('MFCC Mean (Time-Averaged)', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')

plt.subplot(1, 2, 2)
plt.bar(range(n_mfcc), mfcc_std, color='orange')
plt.xlabel('MFCC Coefficient', fontsize=11)
plt.ylabel('Standard Deviation', fontsize=11)
plt.title('MFCC Std (Temporal Variation)', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Feature vector: {n_mfcc * 2} dimensions (mean + std)")
print("This compact representation can be used for classification!")

### Mel Spectrogram vs MFCCs: Comparison

Let's compare these two popular representations side by side.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Mel spectrogram
librosa.display.specshow(mel_spec_db, sr=sr, hop_length=hop_length,
                        x_axis='time', y_axis='mel', cmap='viridis', ax=axes[0])
axes[0].set_title('Mel Spectrogram (128 mel bands)', fontsize=13)
axes[0].set_ylabel('Mel Frequency', fontsize=11)
axes[0].label_outer()

# MFCCs
librosa.display.specshow(mfccs, sr=sr, hop_length=hop_length,
                        x_axis='time', cmap='coolwarm', ax=axes[1])
axes[1].set_title('MFCCs (13 coefficients)', fontsize=13)
axes[1].set_ylabel('MFCC', fontsize=11)
axes[1].set_xlabel('Time (s)', fontsize=11)

plt.tight_layout()
plt.show()

print("Mel spectrogram: More detailed, larger dimension (128 bands)")
print("MFCCs: Compact, decorrelated, fewer dimensions (13 coefficients)")

## 8. Audio Augmentation for Machine Learning

Like image augmentation, we can augment audio to improve model robustness!

**Common techniques:**
- **Time stretching**: Change speed without changing pitch
- **Pitch shifting**: Change pitch without changing speed
- **Adding noise**: Gaussian noise, background noise
- **Time shifting**: Shift audio in time
- **Masking**: Randomly mask time or frequency regions (SpecAugment)

### Time Stretching

Speed up or slow down audio without changing pitch.

In [ ]:
# Time stretch: make it 1.5x faster
y_fast = librosa.effects.time_stretch(y, rate=1.5)

# Time stretch: make it 0.8x slower
y_slow = librosa.effects.time_stretch(y, rate=0.8)

print(f"Original duration: {len(y) / sr:.2f}s")
print(f"Fast (1.5x): {len(y_fast) / sr:.2f}s")
print(f"Slow (0.8x): {len(y_slow) / sr:.2f}s")

### Listen to Time-Stretched Audio

Compare the original with time-stretched versions.

In [ ]:
print("Original:")
display(ipd.Audio(y, rate=sr))

print("\nFast (1.5x):")
display(ipd.Audio(y_fast, rate=sr))

print("\nSlow (0.8x):")
display(ipd.Audio(y_slow, rate=sr))

### Pitch Shifting

Change pitch without changing speed.

In [ ]:
# Pitch shift: up by 4 semitones (musical half-steps)
y_higher = librosa.effects.pitch_shift(y, sr=sr, n_steps=4)

# Pitch shift: down by 3 semitones
y_lower = librosa.effects.pitch_shift(y, sr=sr, n_steps=-3)

print("Pitch shifted audio has same duration but different pitch")
print(f"Duration: {len(y_higher) / sr:.2f}s (same as original)")

### Listen to Pitch-Shifted Audio

Hear how the pitch changes.

In [ ]:
print("Original:")
display(ipd.Audio(y, rate=sr))

print("\nHigher (+4 semitones):")
display(ipd.Audio(y_higher, rate=sr))

print("\nLower (-3 semitones):")
display(ipd.Audio(y_lower, rate=sr))

### Adding Noise

Add Gaussian noise to simulate noisy recording conditions.

In [ ]:
# Add Gaussian noise
noise_factor = 0.005
noise = np.random.randn(len(y)) * noise_factor
y_noisy = y + noise

# Clip to valid range [-1, 1]
y_noisy = np.clip(y_noisy, -1.0, 1.0)

print(f"Original SNR: inf dB (no noise)")
print(f"Noisy SNR: ~{10 * np.log10(np.mean(y**2) / np.mean(noise**2)):.1f} dB")

### Visualizing Augmentations

Compare spectrograms of original vs augmented audio.

In [ ]:
# Compute spectrograms
augmentations = [
    ('Original', y),
    ('Time Stretch (1.5x)', y_fast),
    ('Pitch Shift (+4)', y_higher),
    ('Noisy', y_noisy[:len(y)])  # Trim to same length
]

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes = axes.flatten()

for ax, (title, audio) in zip(axes, augmentations):
    # Compute mel spectrogram
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=2048, 
                                        hop_length=512, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    librosa.display.specshow(mel_db, sr=sr, hop_length=512,
                            x_axis='time', y_axis='mel', cmap='viridis', ax=ax)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Time (s)', fontsize=10)
    ax.set_ylabel('Mel Frequency', fontsize=10)

plt.suptitle('Audio Augmentation Comparison', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

## 9. Building a Simple Audio Classifier

Now let's apply what we've learned to build a simple CNN for audio classification!

We'll use mel spectrograms as inputs (treating them like images) and build a small classifier.

For this example, we'll create synthetic data representing different "classes" of audio.

### Creating Synthetic Audio Dataset

Generate synthetic audio samples with different characteristics for 3 classes.

In [ ]:
def generate_synthetic_audio(class_id, duration=1.0, sr=22050):
    """
    Generate synthetic audio with class-specific characteristics.
    Class 0: Low frequency sine waves
    Class 1: Mid frequency sine waves
    Class 2: High frequency sine waves
    """
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    
    if class_id == 0:
        # Low frequency (200-400 Hz)
        freq = np.random.uniform(200, 400)
    elif class_id == 1:
        # Mid frequency (800-1200 Hz)
        freq = np.random.uniform(800, 1200)
    else:
        # High frequency (2000-3000 Hz)
        freq = np.random.uniform(2000, 3000)
    
    # Generate signal with harmonics
    signal = 0.5 * np.sin(2 * np.pi * freq * t)
    signal += 0.2 * np.sin(2 * np.pi * 2 * freq * t)  # Second harmonic
    
    # Add small random noise
    noise = np.random.randn(len(signal)) * 0.02
    signal += noise
    
    return signal

# Generate dataset
n_samples_per_class = 100
n_classes = 3

print(f"Generating {n_samples_per_class * n_classes} synthetic audio samples...")
print("Class 0: Low frequency (200-400 Hz)")
print("Class 1: Mid frequency (800-1200 Hz)")
print("Class 2: High frequency (2000-3000 Hz)")

### Creating Mel Spectrogram Features

Convert audio samples to mel spectrograms for the CNN.

In [ ]:
# Generate mel spectrograms
X_data = []
y_data = []

for class_id in range(n_classes):
    for _ in range(n_samples_per_class):
        audio = generate_synthetic_audio(class_id)
        
        # Compute mel spectrogram
        mel = librosa.feature.melspectrogram(y=audio, sr=22050, n_fft=1024, 
                                            hop_length=512, n_mels=64)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        X_data.append(mel_db)
        y_data.append(class_id)

X_data = np.array(X_data)
y_data = np.array(y_data)

print(f"Features shape: {X_data.shape}")
print(f"Labels shape: {y_data.shape}")
print(f"(n_samples={len(X_data)}, n_mels={X_data.shape[1]}, time_frames={X_data.shape[2]})")

### Visualizing Sample Spectrograms

Check that our classes have distinct spectral patterns.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for class_id in range(n_classes):
    # Get first sample of this class
    idx = class_id * n_samples_per_class
    mel = X_data[idx]
    
    librosa.display.specshow(mel, sr=22050, hop_length=512,
                            x_axis='time', y_axis='mel', cmap='viridis', ax=axes[class_id])
    axes[class_id].set_title(f'Class {class_id} Sample', fontsize=12)
    axes[class_id].set_xlabel('Time (s)', fontsize=10)
    axes[class_id].set_ylabel('Mel Frequency', fontsize=10)

plt.suptitle('Sample Mel Spectrograms per Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Notice the different frequency distributions!")

### Preparing PyTorch Dataset

Create a PyTorch Dataset and DataLoaders.

In [ ]:
class AudioDataset(Dataset):
    def __init__(self, X, y):
        # Normalize features
        X_normalized = (X - X.mean()) / (X.std() + 1e-8)
        
        # Add channel dimension: (batch, mel, time) -> (batch, 1, mel, time)
        self.X = torch.FloatTensor(X_normalized).unsqueeze(1)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Split data
n_train = int(0.8 * len(X_data))
indices = np.random.permutation(len(X_data))

train_indices = indices[:n_train]
test_indices = indices[n_train:]

train_dataset = AudioDataset(X_data[train_indices], y_data[train_indices])
test_dataset = AudioDataset(X_data[test_indices], y_data[test_indices])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Input shape: {train_dataset[0][0].shape}")

### Building the CNN Classifier

A simple 2D CNN that treats mel spectrograms as images.

In [ ]:
class AudioCNN(nn.Module):
    def __init__(self, n_mels, n_classes):
        super().__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        
        # Global average pooling
        self.gap = nn.AdaptiveAvgPool2d(1)
        
        # Classifier
        self.fc = nn.Linear(128, n_classes)
    
    def forward(self, x):
        # x: (batch, 1, n_mels, time)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        
        # Global average pooling
        x = self.gap(x)  # (batch, 128, 1, 1)
        x = x.view(x.size(0), -1)  # (batch, 128)
        
        x = self.dropout(x)
        x = self.fc(x)
        return x

# Create model
model = AudioCNN(n_mels=64, n_classes=3).to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"\nModel architecture:")
print(model)

### Training the Classifier

Train the model with standard classification setup.

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
n_epochs = 20
train_losses = []
test_accuracies = []

for epoch in range(n_epochs):
    # Training
    model.train()
    epoch_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    # Evaluation
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
    
    accuracy = 100 * correct / total
    test_accuracies.append(accuracy)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{n_epochs}] - Loss: {train_losses[-1]:.4f}, Test Acc: {accuracy:.2f}%")

print(f"\nFinal Test Accuracy: {test_accuracies[-1]:.2f}%")

### Visualizing Training Progress

Plot loss and accuracy curves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
axes[0].plot(train_losses, linewidth=2, color='blue')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Training Loss', fontsize=11)
axes[0].set_title('Training Loss Over Time', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(test_accuracies, linewidth=2, color='green')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[1].set_title('Test Accuracy Over Time', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

### Analyzing Predictions

Look at some predictions to understand what the model learned.

In [ ]:
# Get predictions for test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix', fontsize=14)
plt.xticks([0, 1, 2], ['Low', 'Mid', 'High'])
plt.yticks([0, 1, 2], ['Low', 'Mid', 'High'])

# Add text annotations
for i in range(3):
    for j in range(3):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', 
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=16)

plt.tight_layout()
plt.show()

print("Classes: Low freq (0), Mid freq (1), High freq (2)")

## 10. Modern Approaches: Beyond Hand-Crafted Features

**Traditional pipeline** (what we just did):
1. Raw audio → Hand-crafted features (spectrograms, MFCCs)
2. Features → ML model (CNN, RNN, etc.)

**Modern approaches** learn features end-to-end:

**1. Raw waveform models:**
- **WaveNet**: Dilated convolutions on raw audio
- **SampleRNN**: Hierarchical RNN on samples
- Learn features directly from waveform!

**2. Hybrid approaches:**
- **wav2vec 2.0**: Self-supervised learning on raw audio
- **HuBERT**: Masked prediction on audio
- Pre-train on unlabeled audio, fine-tune on tasks

**3. Transformer-based:**
- **Whisper**: Transformer for speech recognition
- **Audio Spectrogram Transformer (AST)**: Vision Transformer for audio
- Treat spectrograms as "images" for ViT

**Tradeoffs:**
- Hand-crafted features: Interpretable, efficient, domain knowledge
- End-to-end learning: More data needed, potentially better performance, less interpretable

### Comparing Input Representations

Let's visualize the different ways to represent audio for ML.

In [ ]:
# Load short audio clip
y_sample, sr_sample = librosa.load(librosa.example('trumpet'), duration=0.5)

# Compute different representations
# 1. Raw waveform
waveform = y_sample

# 2. Spectrogram
D = librosa.stft(y_sample, n_fft=1024, hop_length=256)
spec = librosa.amplitude_to_db(np.abs(D), ref=np.max)

# 3. Mel spectrogram
mel = librosa.feature.melspectrogram(y=y_sample, sr=sr_sample, n_fft=1024, 
                                     hop_length=256, n_mels=80)
mel_db = librosa.power_to_db(mel, ref=np.max)

# 4. MFCCs
mfcc = librosa.feature.mfcc(y=y_sample, sr=sr_sample, n_mfcc=20, 
                            n_fft=1024, hop_length=256)

# Visualize
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

# Waveform
librosa.display.waveshow(waveform, sr=sr_sample, ax=axes[0], alpha=0.8)
axes[0].set_title('1. Raw Waveform (1D signal)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amplitude', fontsize=10)
axes[0].set_xlabel('')

# Spectrogram
librosa.display.specshow(spec, sr=sr_sample, hop_length=256,
                        x_axis='time', y_axis='hz', cmap='viridis', ax=axes[1])
axes[1].set_title('2. Spectrogram (Linear frequency, 513 bins)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Frequency (Hz)', fontsize=10)
axes[1].set_xlabel('')
axes[1].set_ylim(0, 8000)

# Mel spectrogram
librosa.display.specshow(mel_db, sr=sr_sample, hop_length=256,
                        x_axis='time', y_axis='mel', cmap='viridis', ax=axes[2])
axes[2].set_title('3. Mel Spectrogram (Perceptual frequency, 80 bands)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Mel Frequency', fontsize=10)
axes[2].set_xlabel('')

# MFCCs
librosa.display.specshow(mfcc, sr=sr_sample, hop_length=256,
                        x_axis='time', cmap='coolwarm', ax=axes[3])
axes[3].set_title('4. MFCCs (Compact features, 20 coefficients)', fontsize=12, fontweight='bold')
axes[3].set_ylabel('MFCC', fontsize=10)
axes[3].set_xlabel('Time (s)', fontsize=10)

plt.suptitle('Audio Representations for Machine Learning', fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## 11. Key Takeaways

### Audio vs. Other Modalities
- **Audio** requires both time AND frequency representations
- **Images** are spatial (2D), **text** is symbolic (1D discrete), **audio** is continuous (1D) but best viewed in 2D time-frequency

### Core Transforms
- **FFT**: Time → frequency (loses time information)
- **STFT**: Time → time-frequency (windowed FFT, creates spectrograms)
- **Mel scale**: Linear frequency → perceptual frequency
- **DCT**: Decorrelate mel bands (creates MFCCs)

### When to Use Each Representation

| Representation | Dimensions | Use Case | Pros | Cons |
|---------------|-----------|----------|------|------|
| **Raw waveform** | 1D (time) | End-to-end learning | No information loss | Large input size, needs lots of data |
| **Spectrogram** | 2D (time × freq) | General audio, music | Full frequency detail | Linear scale not perceptual |
| **Mel spectrogram** | 2D (time × mel) | Speech, music | Perceptual scale, good for CNNs | Larger than MFCCs |
| **MFCCs** | 2D (time × coeff) | Speech recognition | Very compact, decorrelated | Lossy, less detail |

### Data Augmentation
- **Time stretching**: Robustness to speaking speed
- **Pitch shifting**: Robustness to different speakers
- **Noise addition**: Robustness to background noise
- **SpecAugment**: Mask time/frequency regions (popular for Transformers)

### Modern Trends
- Moving towards **end-to-end learning** from raw audio
- **Self-supervised pre-training** (wav2vec, HuBERT) very effective
- **Transformers** replacing CNNs/RNNs in many tasks
- Hand-crafted features still useful for:
  - Limited data scenarios
  - Interpretability requirements
  - Computational efficiency

### Practical Advice
1. **Start simple**: Mel spectrograms + CNN is a strong baseline
2. **Domain matters**: Speech vs. music vs. environmental sounds need different approaches
3. **Data augmentation**: Essential for good generalization
4. **Normalization**: Always normalize features (per-sample or per-dataset)
5. **Window size**: Tradeoff between time and frequency resolution
6. **Pre-training**: Use pre-trained models (Whisper, wav2vec) when possible

## Next Steps

Now that you understand audio processing fundamentals, you can:

1. **Explore real datasets**: Speech Commands, ESC-50, GTZAN Music
2. **Build deeper models**: ResNets, Transformers on spectrograms
3. **Try end-to-end learning**: WaveNet, SampleRNN architectures
4. **Use pre-trained models**: Fine-tune Whisper, wav2vec 2.0, HuBERT
5. **Advanced features**: Chroma features, spectral contrast, zero-crossing rate
6. **Sequence models**: RNNs, LSTMs for temporal modeling
7. **Generative models**: GANs, diffusion models for audio synthesis

**Key libraries to explore:**
- `librosa`: Audio analysis and feature extraction
- `torchaudio`: PyTorch audio processing
- `transformers`: Pre-trained audio models (Whisper, Wav2Vec2)
- `audiomentations`: Audio augmentation library